# MD-CDBG Simulation — R-BNE-DO+LCP vs. Asghar et al. (GameSec 2025)

**Thesis:** Multidomain Cyberdeception Game for Resilient IoT Systems  
**Author:** Alioum Abdoulaye — Université Paris-Saclay / Université de Ngaoundéré  
**Supervisors:** Prof. Abdelhak MOURAD (GUEROUI) — Prof. Ado Adamou ABBA ARI

---

## Structure

- **Cell 1**: Install dependencies
- **Cell 2**: LCP Solver (Nash Equilibrium for non-zero-sum bimatrix games)
- **Cell 3**: MD-CDBG Simulation (Algorithm 1: Bayes-DO + Algorithm 2: R-BNE-DO)
- **Cell 4**: Run simulation (10 replications × 3 scenarios × 2 strategies)
- **Cell 5**: Figures (Utility, VC, Resilience, Belief)
- **Cell 6**: Download results

---
**Estimated runtime:** ~10–20 minutes on Colab CPU  
**GPU:** Not needed (pure CPU computation)

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 1 — Dependencies
# ════════════════════════════════════════════════════════════════
!pip install numpy scipy matplotlib -q
import numpy as np
from scipy.optimize import linprog
from itertools import combinations
import matplotlib.pyplot as plt
import matplotlib
import json, time, warnings
warnings.filterwarnings('ignore')
print('Dependencies OK')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 2 — LCP Solver
# Nash Equilibrium for non-zero-sum bimatrix games
# via support enumeration (Porter et al., 2004)
# ════════════════════════════════════════════════════════════════

def _check_support(M_D, M_A, S_D, S_A, tol=1e-8):
    nD, nA = M_D.shape
    S_D = list(S_D); S_A = list(S_A)
    nSD = len(S_D); nSA = len(S_A)

    if nSA == 1:
        sigma_A = np.zeros(nA); sigma_A[S_A[0]] = 1.0
    else:
        A = np.zeros((nSD, nSA + 1))
        for row, i in enumerate(S_D):
            A[row, :nSA] = M_D[i, S_A]; A[row, nSA] = -1.0
        b = np.zeros(nSD)
        A_sum = np.zeros((1, nSA + 1)); A_sum[0, :nSA] = 1.0
        b_sum = np.array([1.0])
        A_full = np.vstack([A, A_sum]); b_full = np.concatenate([b, b_sum])
        try:
            sol, _, _, _ = np.linalg.lstsq(A_full, b_full, rcond=None)
        except np.linalg.LinAlgError:
            return None
        x = sol[:nSA]
        if np.any(x < -tol): return None
        x = np.clip(x, 0, None)
        if abs(x.sum()) < tol: return None
        x /= x.sum()
        sigma_A = np.zeros(nA)
        for j, idx in enumerate(S_A): sigma_A[idx] = x[j]

    payoffs_D = M_D @ sigma_A
    v_D = float(np.mean(payoffs_D[S_D]))
    for i in S_D:
        if abs(payoffs_D[i] - v_D) > tol * 10: return None
    for i in range(nD):
        if i not in S_D and payoffs_D[i] > v_D + tol: return None

    if nSD == 1:
        sigma_D = np.zeros(nD); sigma_D[S_D[0]] = 1.0
    else:
        A2 = np.zeros((nSA, nSD + 1))
        for row, j in enumerate(S_A):
            A2[row, :nSD] = M_A[S_D, j]; A2[row, nSD] = -1.0
        b2 = np.zeros(nSA)
        A2_sum = np.zeros((1, nSD + 1)); A2_sum[0, :nSD] = 1.0
        b2_sum = np.array([1.0])
        A2_full = np.vstack([A2, A2_sum]); b2_full = np.concatenate([b2, b2_sum])
        try:
            sol2, _, _, _ = np.linalg.lstsq(A2_full, b2_full, rcond=None)
        except np.linalg.LinAlgError:
            return None
        y = sol2[:nSD]
        if np.any(y < -tol): return None
        y = np.clip(y, 0, None)
        if abs(y.sum()) < tol: return None
        y /= y.sum()
        sigma_D = np.zeros(nD)
        for i, idx in enumerate(S_D): sigma_D[idx] = y[i]

    payoffs_A = sigma_D @ M_A
    v_A = float(np.mean(payoffs_A[S_A]))
    for j in S_A:
        if abs(payoffs_A[j] - v_A) > tol * 10: return None
    for j in range(nA):
        if j not in S_A and payoffs_A[j] > v_A + tol: return None
    return sigma_D, sigma_A

def solve_bimatrix_support_enum(M_D, M_A, tol=1e-8):
    """Nash Equilibrium via support enumeration (exact, Porter et al. 2004)"""
    nD, nA = M_D.shape
    best = None; best_val = -np.inf
    for s_D_size in range(1, nD + 1):
        for s_A_size in range(1, nA + 1):
            for S_D in combinations(range(nD), s_D_size):
                for S_A in combinations(range(nA), s_A_size):
                    sol = _check_support(M_D, M_A, S_D, S_A, tol)
                    if sol is not None:
                        sD, sA = sol
                        val = float(sD @ M_D @ sA)
                        if val > best_val:
                            best_val = val; best = (sD, sA)
    if best is not None: return best
    # Fallback: minimax LP
    c = np.zeros(nD + 1); c[-1] = -1.0
    A_ub = np.hstack([-M_D.T, np.ones((nA, 1))]); b_ub = np.zeros(nA)
    A_eq = np.ones((1, nD + 1)); A_eq[0, -1] = 0; b_eq = np.array([1.0])
    bounds = [(0, None)] * nD + [(None, None)]
    res = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
    sD = np.ones(nD)/nD
    if res.success:
        sD = np.clip(res.x[:nD], 0, None)
        sD = sD/sD.sum() if sD.sum() > 1e-8 else np.ones(nD)/nD
    return sD, np.ones(nA)/nA

print('LCP Solver OK')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 3 — MD-CDBG Simulation
# Algorithm 1: Bayes-DO | Algorithm 2: R-BNE-DO
# ════════════════════════════════════════════════════════════════

# ── PARAMETERS ──────────────────────────────────────────────────
N_LAYERS = 3
N_A      = [5, 6, 5]    # attacker actions per layer
N_D      = [5, 5, 5]    # defender actions per layer
N_REAL   = [20, 10, 5]  # real nodes per layer
N_DECOY  = [12, 6, 4]   # decoy nodes per layer
T        = 50            # time horizon
N_TYPES  = 3             # θ_opp, θ_pers, θ_state

PRIOR_A  = np.array([0.50, 0.30, 0.20])   # p(θ_A)
BETA_A_L = np.array([0.15, 0.30, 0.55])   # β_k^A attacker layer weights
PSI = np.array([[0.20,0.20,0.20],          # ψ_k(θ_A): detection capability
                [0.50,0.50,0.50],
                [0.65,0.65,0.70]])

ALPHA   = np.array([0.55, 0.32, 0.07, 0.06])  # α_1..α_4 defender utility
BETA_D  = np.array([0.15, 0.30, 0.55])         # β_k^D defender layer weights
BETA_O  = np.array([0.20, 0.30, 0.50])         # β_k^Ω resilience weights
GAMMA_A = np.array([0.55, 0.25, 0.20])         # γ_1..γ_3 attacker utility
OMEGA_W = np.array([0.40, 0.35, 0.25])         # ω_1..ω_3 decoy quality

V_A = [np.array([0.35,0.20,0.15,0.25,0.10]),
       np.array([0.50,0.20,0.45,0.30,0.25,0.40]),
       np.array([0.56,0.55,0.45,0.40,0.50])]
P_A = [np.array([0.28,0.32,0.25,0.38,0.15]),
       np.array([0.25,0.30,0.28,0.35,0.30,0.10]),
       np.array([0.23,0.20,0.25,0.28,0.18])]
R_A_COST = [np.array([2.,1.5,3.,0.5,4.]),
            np.array([3.,1.,4.,5.,2.5,6.]),
            np.array([4.,5.,3.,2.,4.5])]
LAM_KI = [np.array([0.20,0.15,0.10,0.05,0.12]),
          np.array([0.15,0.10,0.20,0.18,0.12,-0.25]),
          np.array([0.08,0.10,0.15,0.08,0.05])]
E_D = [np.array([1980.,240.,150.,300.,120.]),
       np.array([2500.,800.,400.,600.,200.]),
       np.array([1200.,500.,300.,800.,100.])]
C_D = [np.array([8.,2.,1.5,3.,1.]),
       np.array([10.,4.,2.,5.,1.5]),
       np.array([6.,3.,2.5,7.,1.])]
E_MAX = [e.max() for e in E_D]
C_MAX = [c.max() for c in C_D]
R_FUNC = [np.array([0.92,0.72,0.45,0.18,0.28]),
          np.array([0.88,0.70,0.50,0.20,0.30,0.15]),
          np.array([0.90,0.65,0.40,0.15,0.25])]

RHO    = np.array([0.80, 0.90, 0.99])
W_FIX  = np.array([0.50, 0.30, 0.20])
LAM1, LAM2 = 4.62, 4.0
TAU_K  = np.array([5, 3, 2])
TMAX_K = np.array([10, 8, 5])

OMEGA_MIN = 0.55   # Ω_min
DELTA_MIN = 0.60   # δ_min (C2)
BUDGET_B  = 1.2    # B (C3)
GAMMA_D   = 0.95   # discount
ETA0      = 0.10   # step size
MAX_DO    = 20     # max Bayes-DO iterations
ISOLATION = 3      # isolation action index

# ── STATE ───────────────────────────────────────────────────────
class State:
    def __init__(self):
        self.comp   = np.zeros(N_LAYERS, dtype=int)
        self.belief = np.tile(PRIOR_A.copy(), (N_LAYERS, 1))  # μ_k^t(θ_A)
        self.t = 0
    def delta(self, k):  return self.comp[k] / N_REAL[k]
    def delta_bar(self): return float(np.mean([self.delta(k) for k in range(N_LAYERS)]))
    def accessible(self, k): return k == 0 or self.comp[k-1] > 0
    def clone(self):
        s = State(); s.comp=self.comp.copy(); s.belief=self.belief.copy(); s.t=self.t; return s

def _e(n, i):
    v = np.zeros(n); v[i] = 1.; return v

# ── DECOY QUALITY q_k(σ_k^D) ────────────────────────────────────
def decoy_quality(k, sD):
    q_func = float(sD @ R_FUNC[k][:N_D[k]])
    return float(np.clip(OMEGA_W[0]*q_func + OMEGA_W[1]*0.88*q_func + OMEGA_W[2]*0.82*q_func, 0., 1.))

# ── DECEPTION f_1^k ──────────────────────────────────────────────
def f1(k, sA, sD, theta):
    ratio    = N_DECOY[k] / (N_REAL[k] + N_DECOY[k])
    q        = decoy_quality(k, sD)
    psi      = PSI[theta, k]
    d_effort = (float(sA[5]) * psi) if (k == 1 and N_A[k] == 6) else (0.03 * psi)
    d_vuln   = (1. - q) * 0.35
    d        = 1. - (1. - d_effort) * (1. - d_vuln)
    return float(ratio * q * (1. - d))

# ── RESILIENCE Ω_k(t) ────────────────────────────────────────────
def adaptive_weights(dbar):
    w1 = W_FIX[0] * np.exp(-LAM1 * dbar)
    w2 = max(0., W_FIX[1] * LAM2 * np.e * dbar * np.exp(-LAM2 * dbar))
    w3 = max(0., 1. - w1 - w2)
    tot = w1 + w2 + w3
    return (w1/tot, w2/tot, w3/tot) if tot > 1e-8 else tuple(W_FIX)

def omega_local(k, state):
    w1, w2, w3 = adaptive_weights(state.delta_bar())
    d_k = state.delta(k)
    return float(w1*float(1.-d_k>=RHO[k]) + w2*max(0.,1.-d_k) + w3*float(d_k*N_REAL[k]*TAU_K[k]<=TMAX_K[k]))

def omega_global(state):
    return float(sum(BETA_O[k] * omega_local(k, state) for k in range(N_LAYERS)))

# ── COHERENCE ────────────────────────────────────────────────────
def coherence(k, sD_k, sD_k1):
    q_k = decoy_quality(k, sD_k); q_k1 = decoy_quality(k+1, sD_k1)
    return float(1. - abs(q_k - q_k1) / max(q_k, q_k1, 1e-8))

# ── UTILITY FUNCTIONS ────────────────────────────────────────────
def u_D_local(k, sD, sA, theta, state):
    f1k  = f1(k, sA, sD, theta)
    om_k = omega_local(k, state)
    f3k  = float(sD @ np.array(E_D[k][:N_D[k]])) / E_MAX[k]
    f4k  = float(sD @ np.array(C_D[k][:N_D[k]])) / C_MAX[k]
    return float(ALPHA[0]*f1k + ALPHA[1]*om_k - ALPHA[2]*f3k - ALPHA[3]*f4k)

def u_D_bayes(k, sD, sA_dist, belief_k, state):
    return float(sum(
        belief_k[th] * float(sum(sum(sD[i]*sA_dist[th][a]*u_D_local(k,_e(N_D[k],i),_e(N_A[k],a),th,state)
            for a in range(N_A[k])) for i in range(N_D[k])))
        for th in range(N_TYPES)))

def U_D_global(sD_list, sA_dist_list, state):
    return float(sum(BETA_D[k]*u_D_bayes(k,sD_list[k],sA_dist_list[k],state.belief[k],state) for k in range(N_LAYERS)))

def u_A_local(k, sD, sA, theta, state):
    if not state.accessible(k): return 0.0
    f1k = f1(k, sA, sD, theta)
    g   = float(sA @ (V_A[k][:N_A[k]] * (1.-f1k) * P_A[k][:N_A[k]]))
    c   = float(sA @ R_A_COST[k][:N_A[k]]) / R_A_COST[k][:N_A[k]].max()
    rho = float(sA @ (f1k * LAM_KI[k][:N_A[k]]))
    return float(GAMMA_A[0]*g - GAMMA_A[1]*c - GAMMA_A[2]*rho)

def U_A_global(sD_list, sA_list, theta, state):
    return float(sum(
        BETA_A_L[k] * float(sum(sum(sD_list[k][i]*sA_list[k][a]*u_A_local(k,_e(N_D[k],i),_e(N_A[k],a),theta,state)
            for a in range(N_A[k])) for i in range(N_D[k])))
        for k in range(N_LAYERS)))

# ── PAYOFF MATRICES ──────────────────────────────────────────────
def make_MD(k, state, lam, nu_k, belief_k, sD_prev=None):
    nD, nA = N_D[k], N_A[k]
    M = np.zeros((nD, nA))
    for i in range(nD):
        sD_i = _e(nD, i)
        f4k  = float(sD_i @ np.array(C_D[k][:nD])) / C_MAX[k]
        coh_pen = 0.
        if k < N_LAYERS-1 and sD_prev is not None:
            coh = coherence(k, sD_i, sD_prev[k+1]) if k+1 < N_LAYERS else 1.
            coh_pen = nu_k * max(0., DELTA_MIN - coh)
        for j in range(nA):
            sA_j = _e(nA, j)
            u_exp = sum(belief_k[th]*u_D_local(k,sD_i,sA_j,th,state) for th in range(N_TYPES))
            M[i, j] = u_exp - lam*f4k - coh_pen
    return M

def make_MA(k, theta, state):
    nD, nA = N_D[k], N_A[k]
    return np.array([[u_A_local(k,_e(nD,i),_e(nA,j),theta,state) for j in range(nA)] for i in range(nD)])

def zero_sum_gap(k, theta, state, belief_k):
    MD = make_MD(k, state, 0., 0., belief_k)
    MA = make_MA(k, theta, state)
    gap = np.linalg.norm(MD + MA, 'fro'); norm = np.linalg.norm(MD, 'fro')
    return float(gap/norm) if norm > 1e-8 else 0.

# ── ALGORITHM 1: Bayes-DO ────────────────────────────────────────
def bayes_do(k, theta, state, lam, nu_k, belief_k, sD_prev=None):
    """Bayesian Double Oracle — exact local BNE via LCP. No tolerance ε_DO."""
    nD, nA   = N_D[k], N_A[k]
    MD_full  = make_MD(k, state, lam, nu_k, belief_k, sD_prev)
    MA_full  = make_MA(k, theta, state)
    R_D=[0]; R_A=[0]
    sD=_e(nD,0); sA=_e(nA,0)
    for _ in range(MAX_DO):
        MD_r = MD_full[np.ix_(R_D,R_A)]; MA_r = MA_full[np.ix_(R_D,R_A)]
        sD_r, sA_r = solve_bimatrix_support_enum(MD_r, MA_r)
        sD_new=np.zeros(nD); sA_new=np.zeros(nA)
        for ii,d in enumerate(R_D): sD_new[d]=sD_r[ii]
        for jj,a in enumerate(R_A): sA_new[a]=sA_r[jj]
        br_D = int(np.argmax(MD_full @ sA_new))
        br_A = int(np.argmax(sD_new @ MA_full))
        if br_D in R_D and br_A in R_A:
            sD, sA = sD_new, sA_new; break
        if br_D not in R_D: R_D.append(br_D)
        if br_A not in R_A: R_A.append(br_A)
        sD, sA = sD_new, sA_new
    return sD, sA

# ── BAYESIAN UPDATE (Phase C) ────────────────────────────────────
def bayes_update(belief_k, action_idx, k):
    nA = N_A[k]
    prefs = [np.array([0.30,0.25,0.20,0.15,0.10]+[0.]*(nA-5))[:nA],
             np.ones(nA)/nA,
             np.array([0.10,0.15,0.20,0.25,0.30]+[0.]*(nA-5))[:nA]]
    if nA==6: prefs[2]=np.array([0.08,0.10,0.15,0.18,0.22,0.27])
    prefs = [p/p.sum() for p in prefs]
    L = np.array([0.40*p[action_idx]+0.60/nA for p in prefs])+1e-10
    post = L*belief_k; return post/post.sum()

# ── TRANSITION (Phase B) ─────────────────────────────────────────
def transition(state, aA, aD, theta, sD_list, rng):
    s=state.clone(); s.t+=1
    for k in range(N_LAYERS):
        if not s.accessible(k): continue
        f1k = f1(k, _e(N_A[k],aA[k]), sD_list[k], theta)
        if rng.random()<f1k: pass
        elif rng.random()<P_A[k][aA[k]]:
            if s.comp[k]<N_REAL[k]: s.comp[k]+=1
        if aD[k]==ISOLATION and s.comp[k]>0: s.comp[k]-=1
    return s

# ── ALGORITHM 2: R-BNE-DO ────────────────────────────────────────
def run_rbne_do_lcp(theta, seed=0):
    """R-BNE-DO: R-BNE via Bayesian Double Oracle over horizon T."""
    rng=np.random.default_rng(seed); state=State()
    lam=0.; nu=np.zeros(N_LAYERS); nu_omega=0.
    omega_t=np.zeros(T); omega_kt=np.zeros((T,N_LAYERS))
    f1_kt=np.zeros((T,N_LAYERS)); ud_t=np.zeros(T); uA_t=np.zeros(T)
    belief_t=np.zeros((T,N_LAYERS,N_TYPES)); gap_t=np.zeros(T)
    sD_prev=[np.ones(N_D[k])/N_D[k] for k in range(N_LAYERS)]

    for t in range(T):
        for k in range(N_LAYERS): omega_kt[t,k]=omega_local(k,state)
        omega_t[t]=omega_global(state)
        sD_list=[]; sA_list=[]
        resilience_critical=(omega_t[t]<OMEGA_MIN)   # global feasibility check

        # Phase A: Bayes-DO per layer (or fallback)
        for k in range(N_LAYERS):
            if resilience_critical:
                sD=_e(N_D[k],ISOLATION) if state.comp[k]>0 else _e(N_D[k],0)
                sA=np.ones(N_A[k])/N_A[k]
            else:
                sD,sA=bayes_do(k,theta,state,lam,nu[k],state.belief[k],sD_prev)
            sD_list.append(sD); sA_list.append(sA)

        for k in range(N_LAYERS): f1_kt[t,k]=f1(k,sA_list[k],sD_list[k],theta)
        sA_dist_list=[[sA_list[k] if th==theta else np.ones(N_A[k])/N_A[k] for th in range(N_TYPES)] for k in range(N_LAYERS)]
        ud_t[t]=U_D_global(sD_list,sA_dist_list,state)
        uA_t[t]=U_A_global(sD_list,sA_list,theta,state)
        belief_t[t]=state.belief.copy()
        gap_t[t]=float(np.mean([zero_sum_gap(k,theta,state,state.belief[k]) for k in range(N_LAYERS)]))

        # Phase B: action sampling + transition
        aA=[int(rng.choice(N_A[k],p=sA_list[k])) for k in range(N_LAYERS)]
        aD=[int(rng.choice(N_D[k],p=sD_list[k])) for k in range(N_LAYERS)]
        state=transition(state,aA,aD,theta,sD_list,rng)

        # Phase C: Bayesian belief update per layer
        for k in range(N_LAYERS): state.belief[k]=bayes_update(state.belief[k],aA[k],k)

        # Phase D: Lagrange multiplier updates (λ, ν_k, ν_Ω)
        eta=ETA0/np.sqrt(t+1)
        f4t=sum(float(sD_list[k]@np.array(C_D[k][:N_D[k]]))/C_MAX[k] for k in range(N_LAYERS))
        lam=max(0.,lam+eta*(f4t-BUDGET_B))
        for k in range(N_LAYERS-1):
            nu[k]=max(0.,nu[k]+eta*(DELTA_MIN-coherence(k,sD_list[k],sD_list[k+1])))
        nu_omega=max(0.,nu_omega+eta*(OMEGA_MIN-omega_t[t]))
        sD_prev=[s.copy() for s in sD_list]

    w=np.array([GAMMA_D**t for t in range(T)])
    return {'name':'R-BNE-DO+LCP','theta':theta,
            'UD':float(w@ud_t),'UA':float(w@uA_t),'VC':float(np.sum(omega_t<OMEGA_MIN)/T),
            'omega':omega_t.tolist(),'omega_k':omega_kt.tolist(),'f1':f1_kt.tolist(),
            'ud':ud_t.tolist(),'uA':uA_t.tolist(),'belief':belief_t.tolist(),'gap':gap_t.tolist()}

# ── ASGHAR-2025 BASELINE ─────────────────────────────────────────
def run_asghar2025(theta, seed=0):
    """Asghar et al. GameSec 2025: greedy oracle + zero-sum LP."""
    rng=np.random.default_rng(seed); state=State()
    omega_t=np.zeros(T); omega_kt=np.zeros((T,N_LAYERS))
    f1_kt=np.zeros((T,N_LAYERS)); ud_t=np.zeros(T); uA_t=np.zeros(T)
    belief_t=np.tile(PRIOR_A,(T,N_LAYERS,1)).reshape(T,N_LAYERS,N_TYPES); gap_t=np.zeros(T)

    for t in range(T):
        for k in range(N_LAYERS): omega_kt[t,k]=omega_local(k,state)
        omega_t[t]=omega_global(state)
        sD_list=[]; sA_list=[]
        unif=np.ones(N_TYPES)/N_TYPES
        for k in range(N_LAYERS):
            nD=N_D[k]; nA=N_A[k]
            sA_init=np.ones(nA)/nA
            pay=np.array([sum(unif[th]*u_D_local(k,_e(nD,i),sA_init,th,state) for th in range(N_TYPES)) for i in range(nD)])
            top2=np.argsort(-pay)[:min(2,nD)]
            sD=np.zeros(nD); sD[top2[0]]=0.70
            if len(top2)>1: sD[top2[1]]=0.30
            MD_k=make_MD(k,state,0.,0.,unif)
            c2=np.zeros(nA+1); c2[-1]=1.
            res=linprog(c2,A_ub=np.hstack([MD_k,-np.ones((nD,1))]),b_ub=np.zeros(nD),
                        A_eq=np.hstack([np.ones((1,nA)),[[0]]]),b_eq=np.array([1.]),
                        bounds=[(0,None)]*nA+[(None,None)],method='highs')
            sA=np.ones(nA)/nA
            if res.success:
                sA=np.clip(res.x[:nA],0,None); sA=sA/sA.sum() if sA.sum()>1e-8 else np.ones(nA)/nA
            sD_list.append(sD); sA_list.append(sA)

        for k in range(N_LAYERS): f1_kt[t,k]=f1(k,sA_list[k],sD_list[k],theta)
        sA_dist=[[sA_list[k] if th==theta else np.ones(N_A[k])/N_A[k] for th in range(N_TYPES)] for k in range(N_LAYERS)]
        ud_t[t]=U_D_global(sD_list,sA_dist,state); uA_t[t]=U_A_global(sD_list,sA_list,theta,state)
        gap_t[t]=float(np.mean([zero_sum_gap(k,theta,state,unif) for k in range(N_LAYERS)]))
        aA=[int(rng.choice(N_A[k],p=sA_list[k])) for k in range(N_LAYERS)]
        aD=[int(rng.choice(N_D[k],p=sD_list[k])) for k in range(N_LAYERS)]
        state=transition(state,aA,aD,theta,sD_list,rng)

    w=np.array([GAMMA_D**t for t in range(T)])
    return {'name':'Asghar-2025','theta':theta,
            'UD':float(w@ud_t),'UA':float(w@uA_t),'VC':float(np.sum(omega_t<OMEGA_MIN)/T),
            'omega':omega_t.tolist(),'omega_k':omega_kt.tolist(),'f1':f1_kt.tolist(),
            'ud':ud_t.tolist(),'uA':uA_t.tolist(),'belief':belief_t.tolist(),'gap':gap_t.tolist()}

STRATEGIES = [('Asghar-2025', run_asghar2025), ('R-BNE-DO+LCP', run_rbne_do_lcp)]
print('MD-CDBG simulation loaded OK')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 4 — Run simulation
# 10 replications × 3 scenarios × 2 strategies
# ════════════════════════════════════════════════════════════════

N_REP = 10   # increase to 20+ for journal submission
SC_NAMES = ['O (Opportunistic)', 'P (Persistent)', 'E (State-sponsored)']
all_r = {}

t_start = time.time()
for theta in range(3):
    all_r[theta] = {}
    print(f'\n--- Scenario {SC_NAMES[theta]} ---')
    for name, runner in STRATEGIES:
        t0 = time.time()
        reps = [runner(theta, seed=theta*100+r) for r in range(N_REP)]
        all_r[theta][name] = {
            'UD':      [r['UD']      for r in reps],
            'UA':      [r['UA']      for r in reps],
            'VC':      [r['VC']      for r in reps],
            'omega':   [r['omega']   for r in reps],
            'omega_k': [r['omega_k'] for r in reps],
            'f1':      [r['f1']      for r in reps],
            'belief':  [r['belief']  for r in reps],
            'gap':     [r['gap']     for r in reps],
        }
        ud  = np.mean(all_r[theta][name]['UD'])
        ua  = np.mean(all_r[theta][name]['UA'])
        vc  = np.mean(all_r[theta][name]['VC'])
        uds = np.std(all_r[theta][name]['UD'])
        elapsed = time.time()-t0
        print(f'  {name:<16}: U^D={ud:.3f}+/-{uds:.3f}  U^A={ua:.3f}  VC={vc:.3f}  [{elapsed:.0f}s]')

print(f'\nTotal runtime: {(time.time()-t_start)/60:.1f} min')

# Quick summary table
print('\n' + '='*65)
print(f'{"Strategy":<16} {"Sc":>3} {"U^D":>12} {"U^A":>7} {"VC":>6} {"DeltaVC":>8}')
print('='*65)
for theta in range(3):
    vc_a = np.mean(all_r[theta]['Asghar-2025']['VC'])
    vc_r = np.mean(all_r[theta]['R-BNE-DO+LCP']['VC'])
    for name, _ in STRATEGIES:
        d = all_r[theta][name]
        m = np.mean(d['UD']); e = np.std(d['UD'],ddof=1)/np.sqrt(N_REP)*1.96
        dvc = f'-{100*(vc_a-vc_r)/vc_a:.0f}%' if name=='R-BNE-DO+LCP' else ''
        print(f'{name:<16} {["O","P","E"][theta]:>3} {m:>7.3f}+/-{e:.3f} {np.mean(d["UA"]):>7.3f} {np.mean(d["VC"]):>6.3f} {dvc:>8}')
    print()

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 5 — Figures
# ════════════════════════════════════════════════════════════════

COLORS  = {'Asghar-2025':'#27AE60', 'R-BNE-DO+LCP':'#1D3461'}
LS      = {'Asghar-2025':'-.', 'R-BNE-DO+LCP':'-'}
LW      = {'Asghar-2025':2.0, 'R-BNE-DO+LCP':2.8}
LC      = ['#E74C3C', '#3498DB', '#2ECC71']
t_ax    = np.arange(T)
x       = np.arange(3)
w       = 0.28
priors  = [0.50, 0.30, 0.20]

def ci95(data):
    n=len(data); m=np.mean(data); se=np.std(data,ddof=1)/np.sqrt(n)
    return m, se*1.96

# ── FIG 1: U^D and U^A ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, metric, title, flip in [
    (axes[0], 'UD', '(a) Defender utility $U^D_{total}$', False),
    (axes[1], 'UA', '(b) Attacker utility $|U^A_{total}|$ (lower=better for defender)', True)]:
    for si, (name, _) in enumerate(STRATEGIES):
        means=[]; errs=[]
        for t in range(3):
            vals = np.abs(all_r[t][name][metric]) if flip else all_r[t][name][metric]
            m, e = ci95(vals); means.append(m); errs.append(e)
        bars = ax.bar(x+(si-0.5)*w, means, w, yerr=errs, color=COLORS[name],
                      alpha=0.85, label=name, error_kw={'linewidth':1.5,'capsize':4})
        if name == 'R-BNE-DO+LCP':
            for bar in bars: bar.set_edgecolor('#E55B13'); bar.set_linewidth(2.)
    ax.set_xticks(x); ax.set_xticklabels(['Sc O','Sc P','Sc E'])
    ax.set_title(title, fontsize=10); ax.legend(fontsize=9)
    ax.set_ylabel('Utility', fontsize=10); ax.grid(True, axis='y', alpha=0.3)
fig.suptitle('Fig. 1 — MD-CDBG (R-BNE-DO+LCP) vs Asghar et al. (GameSec 2025)\n'
             f'95% CI over {N_REP} runs', fontsize=11)
plt.tight_layout(); plt.savefig('fig1_utility.png', dpi=150, bbox_inches='tight'); plt.show()
print('Fig 1 saved')

# ── FIG 2: VC ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5.5))
for si, (name, _) in enumerate(STRATEGIES):
    means=[]; errs=[]
    for t in range(3):
        m, e = ci95(all_r[t][name]['VC']); means.append(m); errs.append(e)
    bars = ax.bar(x+(si-0.5)*w, means, w, yerr=errs, color=COLORS[name],
                  alpha=0.85, label=name, error_kw={'linewidth':1.5,'capsize':4})
    if name == 'R-BNE-DO+LCP':
        for bar in bars: bar.set_edgecolor('#E55B13'); bar.set_linewidth(2.)
for t in range(3):
    vc_a = np.mean(all_r[t]['Asghar-2025']['VC'])
    vc_r = np.mean(all_r[t]['R-BNE-DO+LCP']['VC'])
    imp  = 100*(vc_a-vc_r)/vc_a
    ax.text(t+0.14, vc_r+0.008, f'-{imp:.0f}%', ha='center', fontsize=9,
            color='#1D3461', fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(['Scenario O','Scenario P','Scenario E'])
ax.set_ylabel('Violation rate VC', fontsize=10)
ax.set_title(f'Fig. 2 — Resilience violations (VC) — {N_REP} runs, 95% CI', fontsize=11)
ax.legend(fontsize=9); ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('fig2_VC.png', dpi=150, bbox_inches='tight'); plt.show()
print('Fig 2 saved')

# ── FIG 3: Resilience trajectories ──────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
SC = ['O (Opportunistic)','P (Persistent)','E (State-sponsored)']
for col in range(3):
    ax = axes[col]
    for name, _ in STRATEGIES:
        mu = np.mean(all_r[col][name]['omega'], axis=0)
        sd = np.std(all_r[col][name]['omega'], axis=0)
        ax.plot(t_ax, mu, lw=LW[name], color=COLORS[name], ls=LS[name], label=name)
        ax.fill_between(t_ax, mu-sd, mu+sd, alpha=0.10, color=COLORS[name])
    ax.axhline(OMEGA_MIN, color='black', lw=1.5, ls='--', label=f'Omega_min={OMEGA_MIN}')
    ax.set_title(SC[col], fontsize=10, fontweight='bold')
    ax.set_xlabel('Time step t'); ax.set_ylim(0., 1.05)
    if col==0: ax.set_ylabel('Omega(t)', fontsize=11)
    ax.grid(True, alpha=0.3)
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, loc='lower center', ncol=3, bbox_to_anchor=(0.5,-0.05), fontsize=9)
fig.suptitle(f'Fig. 3 — Global resilience Omega(t) — {N_REP} runs, shaded=std', fontsize=11, y=1.02)
plt.tight_layout(); plt.savefig('fig3_resilience.png', dpi=150, bbox_inches='tight'); plt.show()
print('Fig 3 saved')

# ── FIG 4: Bayesian belief per layer ────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for col in range(3):
    theta = col; ax = axes[col]
    belief = np.array(all_r[theta]['R-BNE-DO+LCP']['belief'])  # (N_REP, T, N_LAYERS, N_TYPES)
    for k in range(3):
        mu = np.mean(belief[:,:,k,theta], axis=0)
        sd = np.std(belief[:,:,k,theta], axis=0)
        ax.plot(t_ax, mu, lw=1.8, color=LC[k], label=f'mu_{k+1}(theta*)')
        ax.fill_between(t_ax, mu-sd, mu+sd, alpha=0.10, color=LC[k])
    ax.axhline(priors[theta], color='gray', lw=1.2, ls=':', label=f'Prior={priors[theta]}')
    ax.axhline(0.80, color='green', lw=1.2, ls='--', label='ID threshold 0.80')
    ax.set_title(SC[col], fontsize=10, fontweight='bold')
    ax.set_xlabel('t'); ax.set_ylim(0., 1.05)
    if col==0: ax.set_ylabel('mu_k^t(theta_A*)', fontsize=10)
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
fig.suptitle(f'Fig. 4 — Bayesian belief per layer — R-BNE-DO+LCP ({N_REP} runs)', fontsize=11, y=1.02)
plt.tight_layout(); plt.savefig('fig4_belief.png', dpi=150, bbox_inches='tight'); plt.show()
print('Fig 4 saved')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 6 — Save and download results
# ════════════════════════════════════════════════════════════════
from google.colab import files

# Save JSON results
with open('results_md_cdbg.json', 'w') as f:
    json.dump({str(k): v for k, v in all_r.items()}, f)
print('Results saved to results_md_cdbg.json')

# Download all files
for fname in ['fig1_utility.png', 'fig2_VC.png', 'fig3_resilience.png',
              'fig4_belief.png', 'results_md_cdbg.json']:
    files.download(fname)
    print(f'Downloaded: {fname}')